# Bronze: ingestao dos dados brutos

Este notebook le os oito arquivos CSV do Volume e os grava como tabelas Delta no schema `workspace.olist_bronze`, sem aplicar transformacoes. A tipagem e a validacao acontecem na camada Silver.

**Ordem de execucao:** execute este notebook antes do Silver.

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze — inventário de arquivos brutos
# MAGIC Envie os CSVs originais ao Volume antes de executar este notebook.

# COMMAND ----------
from pyspark.sql import Window, functions as F

CATALOG = "workspace"
BRONZE = f"{CATALOG}.olist_bronze"
VOLUME_PATH = "/Volumes/workspace/mvp_ecommerce/mvp_ecommerce/"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")

# Keep Bronze raw: all columns start as strings and are typed in Silver.
tabelas = [
    ("olist_orders_dataset.csv", "orders"),
    ("olist_customers_dataset.csv", "customers"),
    ("olist_products_dataset.csv", "products"),
    ("olist_order_items_dataset.csv", "order_items"),
    ("olist_order_payments_dataset.csv", "payments"),
    ("olist_order_reviews_dataset.csv", "reviews"),
    ("olist_sellers_dataset.csv", "sellers"),
    ("olist_geolocation_dataset.csv", "geolocation"),
]

for csv_file, table_name in tabelas:
    dataframe = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{VOLUME_PATH}{csv_file}")
    )
    dataframe.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(f"{BRONZE}.{table_name}")
    print(f"✅ {BRONZE}.{table_name}: {dataframe.count()} linhas")

✅ workspace.olist_bronze.orders: 99441 linhas
✅ workspace.olist_bronze.customers: 99441 linhas
✅ workspace.olist_bronze.products: 32951 linhas
✅ workspace.olist_bronze.order_items: 112650 linhas
✅ workspace.olist_bronze.payments: 103886 linhas
✅ workspace.olist_bronze.reviews: 104162 linhas
✅ workspace.olist_bronze.sellers: 3095 linhas
✅ workspace.olist_bronze.geolocation: 1000163 linhas
